# 08 — Cash-Flow Forecasting (Model 4)

Reproducible walkthrough of the aggregate cash-flow module. Every number shown here comes from the same
code paths used to build the shipped artifacts:

- `backend/scripts/build_cash_flow_dataset.py` — invoice-clearance target → daily/weekly CSVs
- `backend/scripts/cashflow/harness.py` — chronological folds, metrics, allocation
- `backend/scripts/train_cash_flow_models.py` — candidates, backtest, artifact export

The model forecasts **weekly collections per currency** (USD and CAD independently). Daily values are
derived estimates, never separately trained forecasts. The target is an **invoice-clearance proxy**,
not verified cash received.

Run from the repository root with `backend/.venv/bin/python` (CPython 3.14).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
sys.path.insert(0, str(ROOT / "backend" / "scripts"))

import pandas as pd
from app.cash_flow_target import build_target, weekly_from_daily, weekday_shares

print(ROOT)

## 1. Build the target

The proxy lives in exactly one function. A future payment ledger can replace it without touching
training or serving code.

In [ ]:
source = pd.read_csv(ROOT / "data/processed/flowcred_cleaned.csv")
target, stats = build_target(source, with_stats=True)
weekly = weekly_from_daily(target)
print("daily rows  :", len(target))
print("weekly rows :", len(weekly))
print("totals      :", {k: f"{v:,.0f}" for k, v in stats.totals.items()})
print("zero days   :", stats.zero_days)
print("active days :", stats.active_days)

print("\nweekday shares (training window):")
for currency in ("USD", "CAD"):
    shares = weekday_shares(target, currency)
    print(f"  {currency}:", " ".join(f"{shares[i]:.3f}" for i in range(7)))

## 2. Chronological folds

Validation origins 22–47 (26 folds), holdout 48–59, production origin 60. A forecast at origin `o`
may only read `y[0..o]`.

In [ ]:
from scripts.cashflow import harness as H
from scripts.cashflow.baselines import build_series, load_weekly, load_daily

w = load_weekly(ROOT / "data/processed/cash_flow_weekly.csv")
d = load_daily(ROOT / "data/processed/cash_flow_daily.csv")
series = build_series("USD", w, d)

print("weekly observations:", len(series))
print("first week:", series.week_starts[0], "last week:", series.week_starts[-1])
print("validation origins:", H.FIRST_VALIDATION_ORIGIN, "..", H.LAST_VALIDATION_ORIGIN)
print("holdout origins   :", H.FIRST_HOLDOUT_ORIGIN, "..", H.LAST_HOLDOUT_ORIGIN)
print("production origin :", H.PRODUCTION_ORIGIN)

## 3. Why the due-date baseline is only a 7-day benchmark

Reconstructed leak-free (only invoices issued at or before the origin), the schedule is strong one week
ahead and collapses afterwards, because invoices are issued ~15 days before they fall due.

In [ ]:
from scripts.cashflow.baselines import DueDateForecaster

invoices = pd.read_csv(ROOT / "data/processed/flowcred_cleaned.csv")
model = DueDateForecaster(invoices, "USD")
for o in (30, 35, 40):
    model.fit(series, o)
    one = model.predict(1)[0]
    four = model.predict(4)
    print(f"origin {series.week_starts[o]}  next-week={one:,.0f}  week+4={four[-1]:,.0f}")

## 4. Recorded results

Produced by `train_cash_flow_models.py`. Read from the committed metrics rather than recomputed here.

In [ ]:
comparison = pd.read_csv(ROOT / "reports/cash_flow/model_comparison.csv")
weekly_view = comparison[comparison.scope == "weekly"][["currency", "candidate", "wape", "mae", "rmse", "selected"]]
print(weekly_view.sort_values(["currency", "wape"]).to_string(index=False))

In [ ]:
from app import forecasting

forecasting.load_forecast_models()
print(forecasting.forecast_status()["status"])

payload = forecasting.build_forecast_payload("CAD", 30)
print("asOf", payload["asOf"], "| algorithm", payload["model"]["algorithm"])
print("weekly points:", len(payload["weekly"]), "| daily estimates:", len(payload["daily"]))
print("horizon total:", round(payload["totals"]["predicted"], 2))
print("sum of daily :", round(sum(d["estimated"] for d in payload["daily"]), 2))
assert abs(sum(d["estimated"] for d in payload["daily"]) - payload["totals"]["predicted"]) < 0.01

## 5. Limitations to state in any presentation

1. The target is an invoice-clearance proxy, not verified cash.
2. Data ends 2020-03-13 — every forecast is a historical simulation.
3. Only 61 weekly observations: no annual seasonality, and small parameter budgets.
4. Early 2020 contains a structural break (week of 2020-01-06 collapses), which is why holdout error is
   much higher than validation error for every model, including baselines.
5. 90-day forecasts have no untouched holdout folds; they are validated on the 26 validation origins only.